In [ ]:
import math
import numpy as np
from collections import defaultdict
import re
from heapq import nlargest

# Creation du model

In [ ]:
class NgramLanguageModel:

    def __init__(self):
        # Dictionnaires pour stocker les fréquences des bigrammes et trigrammes
        self.trigram_counts = defaultdict(int)
        self.bigram_counts = defaultdict(int)
        self.k = 0.01  # lissage add-k pour éviter les probabilités nulles

    def prepare_data(self, infile, ngram_size=2, min_count=2):
        # Lecture du fichier d'entrée
        with open(infile, 'r', encoding='utf-8') as file:
            text = file.read()

        # Tokenisation initiale
        tokens = re.findall(r'\b\w+\b', text.lower())

        # Séparation en phrases
        sentences = re.split(r'[.!?]', text)
        processed_tokens = []
        for sentence in sentences:
            if sentence.strip():
                sentence_tokens = re.findall(r'\b\w+\b', sentence.lower())
                # Ajout des tokens de début et fin de phrase
                sentence_tokens = ['<s>'] + sentence_tokens + ['</s>']
                processed_tokens.extend(sentence_tokens)

        # Comptage des occurrences des mots
        word_counts = defaultdict(int)
        for token in processed_tokens:
            word_counts[token] += 1

        # Remplacement des mots rares par <UNK>
        for i, token in enumerate(processed_tokens):
            if word_counts[token] < min_count:
                processed_tokens[i] = '<UNK>'

        return ' '.join(processed_tokens)

    def train(self, infile, ngram_size=2):
        # Prétraitement des données
        corpus = self.prepare_data(infile, ngram_size)
        words = corpus.split()
        self.vocab = set(words)  # vocabulaire

        # Entraînement des n-grammes
        if ngram_size == 2:
            for i in range(len(words) - 1):
                bigram = (words[i], words[i + 1])
                self.bigram_counts[bigram] += 1
        elif ngram_size == 3:
            for i in range(len(words) - 2):
                trigram = (words[i], words[i + 1], words[i + 2])
                self.trigram_counts[trigram] += 1

    def predict_ngram(self, sentence, ngram_size=2):
        # Prédiction de la probabilité logarithmique d'une phrase
        raw_words = sentence.lower().split()
        words = [w if w in self.vocab else '<UNK>' for w in raw_words]

        log_prob = 0.0

        if ngram_size == 2:
            words = ['<s>'] + words + ['</s>']
            for i in range(len(words) - 1):
                bigram = (words[i], words[i + 1])
                count_bigram = self.bigram_counts[bigram]
                count_word = sum(self.bigram_counts.get((words[i], w), 0) for w in list(self.bigram_counts))

                # Calcul de la probabilité avec lissage add-k
                prob = (count_bigram + self.k) / (count_word + self.k * len(self.bigram_counts))
                log_prob += math.log(prob)

        elif ngram_size == 3:
            words = ['<s>', '<s>'] + words + ['</s>']
            for i in range(len(words) - 2):
                trigram = (words[i], words[i + 1], words[i + 2])
                count_trigram = self.trigram_counts[trigram]
                count_bigram = self.bigram_counts[(words[i], words[i + 1])]

                prob = (count_trigram + self.k) / (count_bigram + self.k * len(self.trigram_counts))
                log_prob += math.log(prob)

        return log_prob

    def test_perplexity(self, test_file, ngram_size=2):
        # Calcul de la perplexité du modèle sur un fichier de test
        with open(test_file, "r", encoding="utf-8") as file:
            sentences = file.readlines()

        total_log_prob = 0.0
        total_tokens = 0

        for sentence in sentences:
            # Nettoyage et tokenisation
            words = re.findall(r'\b\w+\b', sentence.lower())

            # Remplacement des mots inconnus
            words = [w if w in self.vocab else '<UNK>' for w in words]
            cleaned_sentence = ' '.join(words)

            # Calcul de la log-probabilité
            log_prob = self.predict_ngram(cleaned_sentence, ngram_size)

            total_log_prob += log_prob
            total_tokens += len(words) + 1  # +1 pour </s>

        # Calcul final de la perplexité
        normalized_log_prob = total_log_prob / total_tokens
        perplexity = math.exp(-normalized_log_prob)

        return perplexity

    def generateText(self, ngram_size=2, max_length=20):
        # Génération de texte basé sur les n-grammes appris
        output_words = ['<s>']

        if ngram_size == 2:
            for _ in range(max_length):
                current_word = output_words[-1]
                # Trouver les bigrammes possibles à partir du mot courant
                possible_bigrams = [(bigram, count) for bigram, count in self.bigram_counts.items() if bigram[0] == current_word]
                if not possible_bigrams:
                    break

                words, counts = zip(*possible_bigrams)
                counts = np.array(counts)

                probabilities = counts / sum(counts)

                next_bigram = np.random.choice(len(words), p=probabilities)
                next_word = words[next_bigram][1]

                if next_word == '</s>':
                    break

                output_words.append(next_word)

        elif ngram_size == 3:
            context = ['<s>', '<s>']
            output_words = ['<s>', '<s>']
            for _ in range(max_length):
                w1, w2 = context[-2], context[-1]
                possible_trigrams = [(trigram, count) for trigram, count in self.trigram_counts.items() if trigram[0] == w1 and trigram[1] == w2]
                if not possible_trigrams:
                    break

                trigrams, counts = zip(*possible_trigrams)
                counts = np.array(counts)

                probabilities = counts / sum(counts)
                next_trigram = np.random.choice(len(trigrams), p=probabilities)
                next_word = trigrams[next_trigram][2]

                if next_word == '</s>':
                    break

                output_words.append(next_word)
                context.append(next_word)

        return ' '.join(output_words[1:])

    def autoComplete(self, text, ngram_size=2):
        # Suggère un mot pour compléter une phrase donnée
        words = text.lower().split()
        if ngram_size == 2:
            last_word = words[-1]
            possible_bigrams = [(bigram, count) for bigram, count in self.bigram_counts.items() if bigram[0] == last_word]
            if not possible_bigrams:
                return None

            words, counts = zip(*possible_bigrams)
            counts = np.array(counts)

            probabilities = counts / sum(counts)

            next_bigram = np.random.choice(len(words), p=probabilities)
            next_word = words[next_bigram][1]

            return next_word

        return None

    def correction(self, word, ngram_size=2):
        # Corrige un mot mal orthographié en utilisant la distance d'édition
        possible_corrections = self.generate_possible_corrections(word)
        scored_corrections = []

        for correction in possible_corrections:
            prob = self.predict_ngram(correction, ngram_size)
            edit_distance = self.levenshtein_distance(word, correction)
            score = prob / (edit_distance + 1)  # score = probabilité pondérée par la distance
            scored_corrections.append((score, correction))

        # Retourne la meilleure correction
        best_correction = max(scored_corrections, key=lambda x: x[0])[1]
        return best_correction

    def generate_possible_corrections(self, word):
        # Génère des variations simples d'un mot (substitutions et suppressions)
        corrections = []
        alphabet = 'abcdefghijklmnopqrstuvwxyz'
        for i in range(len(word)):
            for letter in alphabet:
                correction = word[:i] + letter + word[i+1:]
                corrections.append(correction)
            corrections.append(word[:i] + word[i+1:])  # suppression
        corrections.append(word + 's')  # ajout d'un "s"

        return corrections

    def levenshtein_distance(self, str1, str2):
        # Calcule la distance d'édition (Levenshtein) entre deux mots
        n, m = len(str1), len(str2)
        dp = [[0] * (m + 1) for _ in range(n + 1)]

        for i in range(n + 1):
            for j in range(m + 1):
                if i == 0:
                    dp[i][j] = j
                elif j == 0:
                    dp[i][j] = i
                else:
                    cost = 0 if str1[i - 1] == str2[j - 1] else 1
                    dp[i][j] = min(dp[i - 1][j] + 1,    # suppression
                                   dp[i][j - 1] + 1,    # insertion
                                   dp[i - 1][j - 1] + cost)  # substitution

        return dp[n][m]


# train du model (bigram et trigam) et test avec perplexity

In [ ]:
model1_bigramme = NgramLanguageModel()

model1_bigramme.train("ngramv1.train", ngram_size=2)


print("test preplexity pour bigramme",model1_bigramme.test_perplexity("ngramv1.test"))


test preplexity pour bigramme 0.8144239911976009


In [ ]:
model1_trigramme = NgramLanguageModel()

model1_trigramme.train("ngramv1.train", ngram_size=3)


print("test preplexity pour trigramme",model1_trigramme.test_perplexity("ngramv1.test"))


test preplexity pour trigramme 36.02210392610979


# train du model avec big data.txt et test des fonctions generateText,autoComplete,correction

In [ ]:
model2_bigramme = NgramLanguageModel()

model2_bigramme.train("big_data.txt", ngram_size=2)

print("Texte généré (bigramme) :")
print(model2_bigramme.generateText(ngram_size=2))


text = "hey how"
print("\nComplétion automatique :", model2_bigramme.autoComplete(text))

misspelled_word = "brige"
print("\nCorrection orthographique :", model2_bigramme.correction(misspelled_word))

Texte généré (bigramme) :
rt i revert back whenever i just tried maybe playing

Complétion automatique : is

Correction orthographique : brige
